In [1]:

import sys
import platform
import importlib.util

print("Python version:", sys.version.split()[0])
print("Operating system:", platform.system())

# Check whether TensorFlow is available.
tensorflow_available = (
    importlib.util.find_spec("tensorflow") is not None
)

print("TensorFlow installed:", tensorflow_available)

# Check GPU availability.
if tensorflow_available:
    import tensorflow as tf

    print("TensorFlow version:", tf.__version__)
    print("Available GPUs:", tf.config.list_physical_devices("GPU"))

Python version: 3.13.15
Operating system: Linux
TensorFlow installed: True
TensorFlow version: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
from google.colab import files

uploaded_files = files.upload()

print("\nUploaded file names:")
for file_name in uploaded_files:
    print(file_name)

Saving train.csv to train.csv

Uploaded file names:
train.csv


In [4]:
from pathlib import Path
import pandas as pd

dataset_path = Path("/content/train.csv")

if not dataset_path.exists():
    raise FileNotFoundError(
        "train.csv was not found. Please check the uploaded filename."
    )

print("Dataset found:", dataset_path.exists())
print("File size (MB):", round(dataset_path.stat().st_size / (1024 ** 2), 2))

# Read the dataset and inspect its structure.
news_df = pd.read_csv(dataset_path, on_bad_lines="skip")

print("\nDataset shape:", news_df.shape)
print("Column names:", news_df.columns.tolist())

print("\nFirst two news records:")
display(news_df[["title", "text", "subject", "date"]].head(2))

Dataset found: True
File size (MB): 35.71

Dataset shape: (14991, 4)
Column names: ['title', 'text', 'subject', 'date']

First two news records:


,title,text,subject,date
0,Greens say no support for Macron's EZ budget i...,BERLIN (Reuters) - None of the German parties ...,worldnews,"October 25, 2017"
1,Trump faces uphill battle to overcome court's ...,(Reuters) - U.S. President Donald Trump faces ...,politicsNews,"February 6, 2017"


In [5]:
import pandas as pd

# Keep the original dataset unchanged for comparison.
raw_news_df = news_df.copy()

# Step 1: Remove completely duplicated records.
clean_news_df = raw_news_df.drop_duplicates().copy()

# Step 2: Remove rows where the article text is missing.
clean_news_df = clean_news_df.dropna(subset=["text"]).copy()

# Step 3: Remove articles containing only spaces or empty text.
clean_news_df = clean_news_df[
    clean_news_df["text"].astype(str).str.strip().ne("")
].copy()

# Step 4: Convert article text to strings and normalize whitespace.
clean_news_df["text"] = (
    clean_news_df["text"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.replace("\xad", "", regex=False)
    .str.replace(r"[ \t]+", " ", regex=True)
    .str.strip()
)

# Step 5: Show a transparent cleaning report.
print("DATA CLEANING REPORT")
print("-" * 35)
print("Original rows:", len(raw_news_df))
print("Rows after cleaning:", len(clean_news_df))
print("Rows removed:", len(raw_news_df) - len(clean_news_df))
print("Missing article texts:", clean_news_df["text"].isna().sum())
print(
    "Empty article texts:",
    clean_news_df["text"].str.strip().eq("").sum()
)
print(
    "Duplicate rows remaining:",
    clean_news_df.duplicated().sum()
)

print("\nCleaned article sample:")
print(clean_news_df["text"].iloc[0][:300])

DATA CLEANING REPORT
-----------------------------------
Original rows: 14991
Rows after cleaning: 14892
Rows removed: 99
Missing article texts: 0
Empty article texts: 0
Duplicate rows remaining: 0

Cleaned article sample:
BERLIN (Reuters) - None of the German parties involved in exploratory coalition talks support French President Emmanuel Macron s idea to create a separate budget for the euro zone, a negotiator for the Greens party told Reuters on Wednesday. Reinhard Buetikofer, who participated in a late-night nego


In [6]:
import re

# Combine all cleaned news articles into one text corpus.
# A newline separates one article from the next.
news_text = "\n".join(clean_news_df["text"].astype(str))

# Normalize common formatting issues.
news_text = news_text.replace("\xa0", " ")
news_text = news_text.replace("\xad", "")
news_text = re.sub(r"[ \t]+", " ", news_text)

# Build a consistent vocabulary of unique characters.
vocabulary = sorted(set(news_text))

# Create mappings between characters and numeric IDs.
char_to_id = {
    character: index
    for index, character in enumerate(vocabulary)
}

id_to_char = {
    index: character
    for character, index in char_to_id.items()
}

# Display a transparent data-preparation report.
print("TEXT CORPUS REPORT")
print("-" * 35)
print("Number of articles:", len(clean_news_df))
print("Total characters:", len(news_text))
print("Unique characters:", len(vocabulary))

print("\nFirst 100 characters:")
print(repr(news_text[:100]))

print("\nFirst 15 character mappings:")
for character in vocabulary[:15]:
    print(f"{repr(character)} -> {char_to_id[character]}")

# Verify that every character has a valid mapping.
mapping_is_valid = (
    len(char_to_id) == len(vocabulary)
    and len(id_to_char) == len(vocabulary)
    and all(id_to_char[char_to_id[c]] == c for c in vocabulary)
)

print("\nCharacter mapping verification:", mapping_is_valid)

TEXT CORPUS REPORT
-----------------------------------
Number of articles: 14892
Total characters: 35258965
Unique characters: 128

First 100 characters:
'BERLIN (Reuters) - None of the German parties involved in exploratory coalition talks support French'

First 15 character mappings:
'\n' -> 0
' ' -> 1
'!' -> 2
'"' -> 3
'#' -> 4
'$' -> 5
'%' -> 6
'&' -> 7
"'" -> 8
'(' -> 9
')' -> 10
'*' -> 11
'+' -> 12
',' -> 13
'-' -> 14

Character mapping verification: True


In [7]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

# Display GPU details when a GPU is available.
if tf.config.list_physical_devices("GPU"):
    for gpu in tf.config.list_physical_devices("GPU"):
        print("\nGPU device:", gpu.name)

# Check the size of the encoded corpus without building sequences.
print("\nCorpus characters:", len(news_text))
print("Vocabulary size:", len(vocabulary))

# Estimate how much memory a compact integer encoding would require.
estimated_bytes = len(news_text) * 4

print(
    "Approximate memory for int32 character IDs (GB):",
    round(estimated_bytes / (1024 ** 3), 2)
)

TensorFlow version: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

GPU device: /physical_device:GPU:0

Corpus characters: 35258965
Vocabulary size: 128
Approximate memory for int32 character IDs (GB): 0.13


In [8]:
import numpy as np
import tensorflow as tf

# Training configuration
SEQUENCE_LENGTH = 100
BATCH_SIZE = 64

# Convert the corpus into compact unsigned 8-bit character IDs.
# There are only 128 unique characters, so IDs 0–127 fit in uint8.
encoded_text = np.fromiter(
    (char_to_id[character] for character in news_text),
    dtype=np.uint8,
    count=len(news_text)
)

print("Encoded data type:", encoded_text.dtype)
print("Encoded corpus size (MB):", round(encoded_text.nbytes / (1024 ** 2), 2))

# Use a small sample first; do not create windows for the full corpus yet.
sample_size = 10_000
sample_ids = encoded_text[:sample_size].astype(np.int32)

# Create a dataset of consecutive characters.
sample_dataset = tf.data.Dataset.from_tensor_slices(sample_ids)

# Each example contains 101 characters:
# first 100 are inputs, and the 101st is the target.
sample_dataset = sample_dataset.window(
    SEQUENCE_LENGTH + 1,
    shift=1,
    drop_remainder=True
)

sample_dataset = sample_dataset.flat_map(
    lambda window: window.batch(SEQUENCE_LENGTH + 1)
)

# Separate each window into input characters and next-character targets.
sample_dataset = sample_dataset.map(
    lambda window: (window[:-1], window[-1])
)

# Group examples into batches.
sample_dataset = sample_dataset.batch(BATCH_SIZE)

# Inspect one batch.
for input_batch, target_batch in sample_dataset.take(1):
    print("\nInput batch shape:", input_batch.shape)
    print("Target batch shape:", target_batch.shape)

    # Decode one example to check that the text looks correct.
    example_input = "".join(
        id_to_char[int(char_id)]
        for char_id in input_batch[0].numpy()
    )

    print("\nExample input text:")
    print(repr(example_input[:100]))

    print("\nTarget character:")
    print(repr(id_to_char[int(target_batch[0].numpy())]))

Encoded data type: uint8
Encoded corpus size (MB): 33.63

Input batch shape: (64, 100)
Target batch shape: (64,)

Example input text:
'BERLIN (Reuters) - None of the German parties involved in exploratory coalition talks support French'

Target character:
' '


In [10]:
# Convert to int32 for TensorFlow training operations.
encoded_ids = encoded_text.astype(np.int32)

# Reserve the final 5% of the corpus for validation.
split_index = int(len(encoded_ids) * 0.95)

train_ids = encoded_ids[:split_index]
validation_ids = encoded_ids[split_index:]

print("Training characters:", len(train_ids))
print("Validation characters:", len(validation_ids))

# Build examples without creating one huge overlapping matrix.
def make_next_character_dataset(character_ids, sequence_length, batch_size):
    """Create batches where each input predicts its next character."""

    dataset = tf.data.Dataset.from_tensor_slices(character_ids)

    dataset = dataset.window(
        sequence_length + 1,
        shift=sequence_length + 1,
        drop_remainder=True
    )

    dataset = dataset.flat_map(
        lambda window: window.batch(sequence_length + 1)
    )

    dataset = dataset.map(
        lambda window: (window[:-1], window[-1]),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_dataset = make_next_character_dataset(
    train_ids, SEQUENCE_LENGTH, BATCH_SIZE
)

validation_dataset = make_next_character_dataset(
    validation_ids, SEQUENCE_LENGTH, BATCH_SIZE
)

print("\nDatasets successfully constructed!")

# Inspect one batch to verify shapes instead of calculating len().
for x_batch, y_batch in train_dataset.take(1):
    print("Training input shape:", x_batch.shape)
    print("Training target shape:", y_batch.shape)

Training characters: 33496016
Validation characters: 1762949

Datasets successfully constructed!
Training input shape: (64, 100)
Training target shape: (64,)


In [11]:
import tensorflow as tf

# Make results more reproducible where possible.
tf.keras.utils.set_random_seed(42)

VOCAB_SIZE = len(vocabulary)
EMBEDDING_DIM = 64
LSTM_UNITS = 128

# Define the model architecture.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQUENCE_LENGTH,)),

    # Convert character IDs into learned vector representations.
    tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    # Learn patterns in the sequence of characters.
    tf.keras.layers.LSTM(LSTM_UNITS),

    # Predict probabilities for all possible next characters.
    tf.keras.layers.Dense(VOCAB_SIZE)
])

# Configure training with a suitable optimizer and loss.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["sparse_categorical_accuracy"]
)

# Display the architecture and parameter counts.
model.summary()

# Test the model on one batch without updating its weights.
for x_batch, y_batch in train_dataset.take(1):
    predictions = model(x_batch, training=False)

    print("\nInput batch shape:", x_batch.shape)
    print("Target batch shape:", y_batch.shape)
    print("Prediction shape:", predictions.shape)

    # The predicted class is not expected to be accurate yet.
    predicted_id = int(tf.argmax(predictions[0]).numpy())

    print("Example predicted character before training:",
          repr(id_to_char[predicted_id]))
    break

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 123,520 (482.50 KB)

 Trainable params: 123,520 (482.50 KB)

 Non-trainable params: 0 (0.00 B)


Input batch shape: (64, 100)
Target batch shape: (64,)
Prediction shape: (64, 128)
Example predicted character before training: 'w'


In [19]:
# Purpose: Learn character patterns from news text and monitor validation performance.

import time
import tensorflow as tf

# Stop when validation loss stops improving.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Save the best model weights during training.
best_weights_path = "/content/best_news_lstm.weights.h5"

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=best_weights_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)

training_time = time.time() - start_time

print("\nTRAINING SUMMARY")
print("-" * 35)
print("Epochs completed:", len(history.history["loss"]))
print("Training time (minutes):", round(training_time / 60, 2))
print("Final training loss:", round(history.history["loss"][-1], 4))
print("Final validation loss:", round(history.history["val_loss"][-1], 4))
print(
    "Final training accuracy:",
    round(history.history["sparse_categorical_accuracy"][-1], 4)
)
print(
    "Final validation accuracy:",
    round(history.history["val_sparse_categorical_accuracy"][-1], 4)
)
print("Best model weights:", best_weights_path)


Epoch 1/10
   5180/Unknown 152s 29ms/step - loss: 2.4186 - sparse_categorical_accuracy: 0.3158
Epoch 1: val_loss improved from None to 2.03449, saving model to /content/best_news_lstm.weights.h5

Epoch 1: finished saving model to /content/best_news_lstm.weights.h5
5181/5181 ━━━━━━━━━━━━━━━━━━━━ 159s 31ms/step - loss: 2.2519 - sparse_categorical_accuracy: 0.3573 - val_loss: 2.0345 - val_sparse_categorical_accuracy: 0.4149
Epoch 2/10
5181/5181 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 1.9577 - sparse_categorical_accuracy: 0.4372
Epoch 2: val_loss improved from 2.03449 to 1.84773, saving model to /content/best_news_lstm.weights.h5

Epoch 2: finished saving model to /content/best_news_lstm.weights.h5
5181/5181 ━━━━━━━━━━━━━━━━━━━━ 156s 30ms/step - loss: 1.9089 - sparse_categorical_accuracy: 0.4502 - val_loss: 1.8477 - val_sparse_categorical_accuracy: 0.4673
Epoch 3/10
5181/5181 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 1.7895 - sparse_categorical_accuracy: 0.4848
Epoch 3: val_loss improved

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 8: val_loss improved from 1.60028 to 1.58489, saving model to /content/best_news_lstm.weights.h5

Epoch 8: finished saving model to /content/best_news_lstm.weights.h5
5181/5181 ━━━━━━━━━━━━━━━━━━━━ 166s 32ms/step - loss: 1.5162 - sparse_categorical_accuracy: 0.5599 - val_loss: 1.5849 - val_sparse_categorical_accuracy: 0.5415
Epoch 9/10
   3/5181 ━━━━━━━━━━━━━━━━━━━━ 2:27 29ms/step - loss: 1.5811 - sparse_categorical_accuracy: 0.5842

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


5180/5181 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.4956 - sparse_categorical_accuracy: 0.5662

...

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


5181/5181 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.4759 - sparse_categorical_accuracy: 0.5712

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 10: val_loss improved from 1.57327 to 1.56275, saving model to /content/best_news_lstm.weights.h5

Epoch 10: finished saving model to /content/best_news_lstm.weights.h5
5181/5181 ━━━━━━━━━━━━━━━━━━━━ 202s 39ms/step - loss: 1.4746 - sparse_categorical_accuracy: 0.5710 - val_loss: 1.5627 - val_sparse_categorical_accuracy: 0.5499

TRAINING SUMMARY
-----------------------------------
Epochs completed: 10
Training time (minutes): 29.6
Final training loss: 1.4746
Final validation loss: 1.5627
Final training accuracy: 0.571
Final validation accuracy: 0.5499
Best model weights: /content/best_news_lstm.weights.h5


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [21]:

# Generate news-style text with the trained LSTM model

def generate_news_text(start_text, characters_to_generate=300, temperature=0.8):
    """
    Generate text one character at a time using the trained LSTM.

    start_text: Prompt that guides the generated text.
    characters_to_generate: Number of new characters to generate.
    temperature: Controls randomness; lower values are more predictable.
    """

    if not isinstance(start_text, str) or not start_text.strip():
        raise ValueError("Please provide a non-empty starting text.")

    if not isinstance(characters_to_generate, int) or characters_to_generate < 0:
        raise ValueError("characters_to_generate must be a non-negative integer.")

    if temperature <= 0:
        raise ValueError("temperature must be greater than zero.")

    supported_prompt = "".join(
        character for character in start_text
        if character in char_to_id
    )

    if not supported_prompt:
        raise ValueError("The prompt contains no supported characters.")

    generated_text = supported_prompt

    for _ in range(characters_to_generate):
        context = generated_text[-SEQUENCE_LENGTH:].rjust(SEQUENCE_LENGTH)

        input_ids = np.array(
            [[char_to_id[character] for character in context]],
            dtype=np.int32
        )

        logits = model(input_ids, training=False)[0]
        probabilities = tf.nn.softmax(logits / temperature).numpy()

        next_id = np.random.choice(len(probabilities), p=probabilities)
        generated_text += id_to_char[int(next_id)]

    return generated_text


# Generate a sample .
np.random.seed(42)

news_prompt = "BERLIN (Reuters) - "

generated_news = generate_news_text(
    start_text=news_prompt,
    characters_to_generate=300,
    temperature=0.8
)

print("GENERATED NEWS TEXT")
print("-" * 50)
print(generated_news)


GENERATED NEWS TEXT
--------------------------------------------------
BERLIN (Reuters) - President Atthe Jerland President to and importer with the be that this 50-year than front state, Cain are she askept to the could be up this killed by Mixand and Raper and Iran than the government, that s as a state, is negory to meeting attin throw it down any out. En number strike North Committee


In [22]:
import os

checkpoint = "/content/best_news_lstm.weights.h5"

if os.path.exists(checkpoint):
    print("Checkpoint mil gaya:", checkpoint)
else:
    print("Checkpoint is location par nahi mila.")

Checkpoint mil gaya: /content/best_news_lstm.weights.h5


In [23]:
from google.colab import files

files.download("/content/best_news_lstm.weights.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>